# The last-success problem and Bruss's odds theorem

Observe independent Bernoulli variables $I_1,\ldots,I_n$, where $\Pr(I_k=1)=p_k$. You want to stop on the **last success**, not merely any success. Once a success is accepted, you cannot change your decision.

Assume $0\le p_k<1$ and that probabilities are known. The optimal rule begins accepting successes only at a particular index $s$. This is **not** the classical best-candidate secretary problem.


## Mathematical rule

Define the odds $o_k=p_k/(1-p_k)$. Scanning backward, find the largest index $s$ such that $\sum_{j=s}^n o_j\ge 1$. If the total odds sum is below $1$, use $s=1$. Reject successes before $s$ and take the first success at or after $s$.

For any starting index $s$, the exact success probability is the chance that **exactly one** success occurs in the suffix $s,\ldots,n$:

$$
P(s)=\sum_{k=s}^n p_k\!\prod_{\substack{j=s\\j\ne k}}^n(1-p_j).
$$


In [ ]:
from fractions import Fraction
import random
from itertools import product

p = [Fraction("0.20"), Fraction("0.50"), Fraction("0.30"), Fraction("0.40")]

def odds_start(p):
    if not p or any(x < 0 or x >= 1 for x in p):
        raise ValueError("Every probability must lie in [0,1).")
    tail_odds = Fraction(0)
    for k in range(len(p)-1, -1, -1):
        tail_odds += p[k] / (1-p[k])
        if tail_odds >= 1:
            return k + 1  # 1-based
    return 1

def exact_win_probability(p, start):
    if not 1 <= start <= len(p):
        raise ValueError("Start must be between 1 and n.")
    return sum(
        (p[k] * math_product((1-p[j]) for j in range(start-1, len(p)) if j != k)
         for k in range(start-1, len(p))),
        Fraction(0)
    )

def math_product(values):
    result = Fraction(1)
    for value in values:
        result *= value
    return result

s = odds_start(p)
scores = [exact_win_probability(p, i) for i in range(1, len(p)+1)]
print("Optimal start:", s)
for k, value in enumerate(scores, 1):
    print(f"Start {k}: P={float(value):.4%}")
assert scores[s-1] == max(scores)


## Simulate sequential observations

For each run, sample a Bernoulli outcome at every time using a seeded random generator. The stopping rule accepts the first observed success from index $s$ onward. It wins exactly when no later successes occur.

The simulation is evaluated **after generating the sequence**, but the stopping decision only uses outcomes already observed.


In [ ]:
def simulate_last_success(p, start, trials=20000, seed=2026):
    if trials <= 0 or not 1 <= start <= len(p):
        raise ValueError("Invalid trials or start.")
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        outcomes = [rng.random() < float(pi) for pi in p]
        selected = next((k for k in range(start-1, len(p)) if outcomes[k]), None)
        wins += selected is not None and not any(outcomes[selected+1:])
    return wins / trials

for k in range(1, len(p)+1):
    observed = simulate_last_success(p, k)
    print(f"Start {k}: simulation {observed:.3%}, exact {float(scores[k-1]):.3%}")


## Visual comparison

Each bar below is the exact winning probability of a fixed start index. Compare them with the simulated frequencies printed above.


In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.bar(list(range(1,len(p)+1)), [float(x) for x in scores])
    plt.axvline(s, linestyle="--", label=f"Sum-the-odds start = {s}")
    plt.xlabel("First eligible trial (s)")
    plt.ylabel("Probability of catching the last success")
    plt.xticks(list(range(1,len(p)+1)))
    plt.legend()
    plt.show()
except ImportError:
    print("Install matplotlib to view the optional figure.")


## Exercises

1. Change the probabilities to $0.1,0.1,0.1,0.1$. Why is $s=1$?
2. Compare the exact and simulated probabilities when all $p_k=0$.
3. Increase one of the success probabilities to $0.9$. Does the optimal start change?
4. Explain why accepting the **first** success does not necessarily maximize the chance of selecting the **last** success.

**Reference:** F. T. Bruss, *Sum the Odds to One and Stop* (2000).
